<a href="https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1:
The research suggests that some content and search signals can be useful for identifying pages that may need attention. My question is whether the labels used for this finding represent a directly observed outcome or a rule-defined outcome.

Finding 2:
The research suggests that combining multiple signals can improve content prioritization. My question is whether the validation method tests the model on data that is sufficiently different from the training data.

Methodology questions:
I would like to understand how the labels were created, whether the data contains repeated pages or clients, and whether the validation split prevents information from the same client or time period from appearing in both training and testing.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import os
import subprocess
import sys
import pandas as pd
import numpy as np

# Go to /content
os.chdir("/content")

# Clone your GitHub repository if it is not already there
repo = "/content/flyrank-ml-internship"

if not os.path.exists(repo):
    subprocess.run([
        "git", "clone",
        "https://github.com/shweta-1202/flyrank-ml-internship.git",
        repo
    ], check=True)

# Find the dataset
possible_paths = [
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv",
    "/content/flyrank-ml-internship/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv",
    "/content/flyrank-ml-internship/data/content_refresh_anonymized.csv"
]

file_path = None

for path in possible_paths:
    if os.path.exists(path):
        file_path = path
        break

# If still not found, search the repository
if file_path is None:
    for root, dirs, files in os.walk(repo):
        if "content_refresh_anonymized.csv" in files:
            file_path = os.path.join(
                root,
                "content_refresh_anonymized.csv"
            )
            break

if file_path is None:
    print("❌ Dataset was not found.")
    print("\nRepository folders:")
    for item in os.listdir(repo):
        print(item)
    raise FileNotFoundError(
        "content_refresh_anonymized.csv is not available in your GitHub repository."
    )

# Load data
df = pd.read_csv(file_path)

print("✅ Dataset loaded successfully!")
print("File:", file_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

✅ Dataset loaded successfully!
File: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

y = df["is_declining"]

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]

top50 = np.argsort(-scores)[:50]

precision_50 = y_test.iloc[top50].mean()

print("Grouped/client-holdout Precision@50:",
      round(precision_50, 3))

Grouped/client-holdout Precision@50: 0.72


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I checked my final feature set for leakage. My target is based on trend_direction, so I do not use trend_direction or trend_pct as model features.

I also avoid product decision flags and future outcome information. The features used by my model are signals that can be observed before making a content refresh recommendation.

Therefore, the model is intended to support a decision rather than use the answer as an input.

In [4]:
leakage_columns = [
    "trend_direction",
    "trend_pct"
]

print("Final model features:")
print(features)

print("\nLeakage check:")

for column in leakage_columns:
    print(
        column,
        "USED" if column in features else "NOT USED"
    )

Final model features:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

Leakage check:
trend_direction NOT USED
trend_pct NOT USED


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Strong claim:
"My model predicts which pages need to be updated."

Safer claim:
"My model provides decision-support by ranking pages that appear more likely to need content review based on the available search and content signals."

The result is observed and measured on the available dataset. It is directional and should be reviewed by a human before taking action. It does not prove causation or predict Google's algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.